In [ ]:
# Instalar kaleido para exportar imágenes
!pip install -U kaleido

import h5py
import numpy as np
import umap
import plotly.graph_objs as go

# Ruta al archivo (ajusta si subes desde archivos)
ruta = "EmbeddingsRawDS[716]&AmyAVariants.h5"

# Definir categorías generales
categorias = {
    "Hidrolasas Archaea": ["Ha"],
    "Hidrolasas Bacterianas": ["Hb"],
    "Hidrolasas Eucariotas": ["He"],
    "Transferasas Archaea": ["Ta"],
    "Transferasas Bacterianas": ["Tb"],
    "Transferasas Eucariotas": ["Te"],
    "AmyA": ["AmyA"]
}

# Cargar datos y etiquetas
grupos = {}
claves = []
etiquetas = []

with h5py.File(ruta, 'r') as archivo:
    for clave in archivo.keys():
        if isinstance(archivo[clave], h5py.Dataset):
            grupos[clave] = archivo[clave][:]
            claves.append(clave)
            for categoria, subgrupos in categorias.items():
                if any(clave.startswith(subgrupo) for subgrupo in subgrupos):
                    etiquetas.append(categoria)
                    break

X = np.vstack([np.array(grupos[clave]) for clave in grupos])

print(f"Total de embeddings cargados: {len(etiquetas)}")
print("Forma de X:", X.shape)

# Reducir a 2D con UMAP
reductor = umap.UMAP(
    n_components=2,
    metric='cosine',
    n_neighbors=60,
    min_dist=0.1,
    random_state=42
)
X_2d = reductor.fit_transform(X)

# Colores por categoría
colores = {
    "Hidrolasas Archaea": "#000080",
    "Hidrolasas Bacterianas": "#DC143C",
    "Hidrolasas Eucariotas": "#FFFF00",
    "Transferasas Archaea": "#FFA500",
    "Transferasas Bacterianas": "#006400",
    "Transferasas Eucariotas": "#800080",
    "AmyA": "#000000"
}

# Crear visualización
trazas = []
for categoria in categorias.keys():
    indices = [i for i, etiqueta in enumerate(etiquetas) if etiqueta == categoria]
    textos = [claves[i] for i in indices]
    traza = go.Scatter(
        x=X_2d[indices, 0],
        y=X_2d[indices, 1],
        mode='markers',
        name=categoria,
        text=textos,
        hoverinfo='text+name',
        marker=dict(
            size=6,
            color=colores[categoria],
            opacity=0.75,
            line=dict(width=0.5, color='black')
        )
    )
    trazas.append(traza)

fig = go.Figure(data=trazas)

fig.update_layout(
    title='Proyección UMAP 2D con Agrupación por Categorías',
    xaxis_title='UMAP 1',
    yaxis_title='UMAP 2',
    legend_title='Categorías Enzimáticas',
    margin=dict(l=0, r=0, b=0, t=30),
    width=900,
    height=800
)

# Mostrar interactivo
fig.show()

# Exportar como HTML interactivo
fig.write_html("UMAP2D_Categorized_Embeddings.html")

# Exportar imagen en alta resolución
fig.write_image("UMAP2D_Categorized_Embeddings.png", scale=4, width=1200, height=1000)

# Exportar también como SVG vectorial
fig.write_image("UMAP2D_Categorized_Embeddings.svg")

print("Visualización guardada como HTML, PNG y SVG.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 8.5 MB/s eta 0:00:00
Total de embeddings cargados: 695
Forma de X: (702, 1280)


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Visualización guardada como HTML, PNG y SVG.
